# Check A: is the gaze mapping even right?  Check B: does FRM have any headroom?

Self-contained — regenerates the leave-one-out ground truth and the sink mask if they are not
already on Drive (~1 min for 20 examples, plus model load).

## A — aspect ratio

The token grid is **9x9 square**. WEAR-VQA frames are **portrait**. `gaze_patch` maps
`x_norm, y_norm` straight onto the square grid with **no letterbox correction**.

If SmolVLM **pads** the frame to square rather than squashing it, the image occupies only the middle
columns and **every gaze->patch mapping is wrong horizontally**.

That matters because Check 2 — our best result — reported offsets spread sideways
(col sigma 2.69 vs row sigma 1.72) and I read that as "context sits at eye level, displaced
horizontally". **A left/right pad produces exactly that signature artificially.** This either
confirms the finding or kills it.

## B — headroom

The FRM spec's make-or-break is Exp 1: **FRM vs pure eccentricity. If FRM ~ eccentricity, drop
Stage 2b.** So before generating labels or training anything, measure what a *geometric rule* can
already do: fit an anisotropic Gaussian around the gaze using Check 2's measured offset spread and
score it on the same referee.

| predictor | precision@10 |
|---|---|
| attn x grad (best teacher) | 35.0% |
| attention (`imp_a`) | 31.1% |
| **anisotropic gaze blob** | **?** |
| isotropic gaze proximity | 19.4% |
| CTRL random | 13.9% |

*(the two teacher numbers are quoted from the bake-off; they are scored against the same drops and
involve no gaze, so the aspect-ratio question does not affect them)*

**~33% -> a 3-parameter geometric rule matches the best teacher. FRM has no headroom; the spec's own
kill criterion fires.** **~20-22% -> real room for a learned module; proceed to LOO-at-scale.**

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, glob, json, time
import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

DATA_ROOT = "/content/drive/MyDrive/wearvqa_gaze_only"
CACHE     = "/content/drive/MyDrive/wearvqa_faithfulness.pt"
SINKF     = "/content/drive/MyDrive/sink_mask_smolvlm2.pt"
MODEL_ID  = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
N_PER_TYPE, GROUP = 2, 1
assert os.path.isdir(DATA_ROOT), f"dataset not found at {DATA_ROOT}"

model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer
print("device:", device)

## 2. Ground truth — load from Drive, or regenerate

Ablate each patch (`attention_mask -> 0`), measure the drop in `log P(gold answer)`. That drop is
the ground-truth importance of the patch.

In [ ]:
def build_inputs(image, question, answer):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    tok = lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)
    return float(tok[n_prompt - 1:].sum())

def patch_groups(L_v, g):
    G_ = int(round(math.sqrt(L_v)))
    if g <= 1:
        return [[i] for i in range(L_v)]
    return [[r * G_ + c for r in range(r0, min(r0 + g, G_)) for c in range(c0, min(c0 + g, G_))]
            for r0 in range(0, G_, g) for c0 in range(0, G_, g)]

def collect_samples():
    types = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
    out = []
    for t in types:
        for jp in sorted(glob.glob(os.path.join(DATA_ROOT, t, "*.json")))[:N_PER_TYPE]:
            m = json.load(open(jp)); ip = jp[:-5] + ".jpg"
            if os.path.exists(ip) and "gaze" in m and m.get("response"):
                out.append(dict(type=t, img_path=ip, question=m["question"],
                                answer=m["response"], gaze=m["gaze"]))
    return out

if os.path.exists(CACHE):
    data = torch.load(CACHE, weights_only=False)
    print(f"loaded {len(data)} cached examples")
else:
    print("cache not found - regenerating the LOO ground truth")
    samples, data, t0 = collect_samples(), [], time.time()
    for i, s in enumerate(samples):
        img = S.load_image(s["img_path"])
        inp, n_prompt = build_inputs(img, s["question"], s["answer"])
        ids = inp["input_ids"][0].cpu()
        iid = S._find_image_token_id(model, processor)
        img_pos = torch.nonzero(ids == iid).squeeze(-1)
        base = answer_logprob(inp, n_prompt)
        drops = torch.zeros(len(img_pos))
        for grp in patch_groups(len(img_pos), GROUP):
            am = inp["attention_mask"].clone(); am[0, img_pos[grp]] = 0
            drops[grp] = (base - answer_logprob(inp, n_prompt, am)) / len(grp)
        data.append(dict(**s, base_logp=base, drops=drops))
        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(samples)}  ({(time.time()-t0)/60:.1f} min)")
    torch.save(data, CACHE)
    print(f"saved -> {CACHE}")

L_v = data[0]["drops"].numel(); G = int(round(math.sqrt(L_v))); N = len(data)

if os.path.exists(SINKF):
    sinks = torch.load(SINKF, weights_only=False)["sink_mask"].bool()
else:
    print("sink mask not found - detecting from 6 examples")
    sc = []
    for d in data[:6]:
        o = S.make_smolvlm_output(image=S.load_image(d["img_path"]), question=d["question"])
        sc.append(VS.sink_scores(o.post_softmax, o.image_token_mask, o.text_token_mask,
                                 is_post_softmax=True)); del o
    score = VS.aggregate_sink_scores(sc); sinks = VS.detect_sinks(score)
    torch.save({"model": MODEL_ID, "L_v": L_v, "sink_mask": sinks, "sink_score": score}, SINKF)
    print(VS.sink_report(score, sinks))

cand     = VS.candidate_mask(L_v, exclude=[sinks])
cand_idx = torch.nonzero(cand, as_tuple=False).squeeze(-1)
n_cand   = int(cand.sum())
print(f"\n{N} examples | L_v={L_v} ({G}x{G}) | candidates {n_cand} | chance@10 {10/n_cand:.1%}")

## 3. CHECK A — does the processor pad or squash?

In [ ]:
img  = S.load_image(data[0]["img_path"])
W, H = img.size
msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "x"}]}]
enc  = processor(text=processor.apply_chat_template(msgs, add_generation_prompt=True),
                 images=[img], return_tensors="pt")

pv = enc["pixel_values"]
while pv.ndim > 4:
    pv = pv[0]
_, ph, pw = pv.shape[-3:]
print(f"source frame      : {W} x {H}   aspect {W/H:.3f}  ({'portrait' if H > W else 'landscape'})")
print(f"pixel_values      : {tuple(pv.shape)}  -> {pw} x {ph}, aspect {pw/ph:.3f}")

box = None
if "pixel_attention_mask" in enc:                       # Idefics3 tells us directly
    m = enc["pixel_attention_mask"]
    while m.ndim > 2:
        m = m[0]
    rows = torch.nonzero(m.any(dim=1)).squeeze(-1); cols = torch.nonzero(m.any(dim=0)).squeeze(-1)
    box = (int(cols[0]), int(cols[-1]) + 1, int(rows[0]), int(rows[-1]) + 1)
    print(f"pixel_attention_mask present: valid region {box[1]-box[0]} x {box[3]-box[2]}")
else:                                                   # fall back: constant border detection
    v = pv[0] if pv.ndim == 3 else pv
    colvar = v.float().std(dim=(0, 1)); rowvar = v.float().std(dim=(0, 2))
    nzc = torch.nonzero(colvar > 1e-6).squeeze(-1); nzr = torch.nonzero(rowvar > 1e-6).squeeze(-1)
    if len(nzc) and len(nzr):
        box = (int(nzc[0]), int(nzc[-1]) + 1, int(nzr[0]), int(nzr[-1]) + 1)
    print(f"no pixel_attention_mask; non-constant region {box[1]-box[0]} x {box[3]-box[2]}")

x0f, x1f, y0f, y1f = 0.0, 1.0, 0.0, 1.0
PADDED = False
if box:
    x0f, x1f, y0f, y1f = box[0]/pw, box[1]/pw, box[2]/ph, box[3]/ph
    content_aspect = (box[1]-box[0]) / max(box[3]-box[2], 1)
    PADDED = (x1f - x0f) < 0.98 or (y1f - y0f) < 0.98
    print(f"content occupies  : x [{x0f:.3f}, {x1f:.3f}]   y [{y0f:.3f}, {y1f:.3f}]")
    print(f"content aspect    : {content_aspect:.3f}   vs source {W/H:.3f}")

print()
if PADDED:
    print("*** PADDED — the frame does NOT fill the square grid.")
    print("    Every gaze->patch mapping that ignores this is WRONG. Corrected below.")
else:
    print("*** SQUASHED (or already square) — the frame fills the grid.")
    print("    The existing gaze->patch mapping is correct; Check 2 stands as reported.")

## 4. Check 2, recomputed with the correct mapping

In [ ]:
def gaze_patch_raw(gz):                       # what every previous notebook used
    return min(G-1, int(gz["y_norm"]*G)) * G + min(G-1, int(gz["x_norm"]*G))

def gaze_patch_fixed(gz):                     # letterbox-aware
    xc = x0f + gz["x_norm"] * (x1f - x0f)
    yc = y0f + gz["y_norm"] * (y1f - y0f)
    return min(G-1, int(yc*G)) * G + min(G-1, int(xc*G))

def summarise(fn, label):
    dist, offs, moved = [], [], 0
    for d in data:
        if float(d["drops"][cand_idx].max()) <= 1e-6:
            continue
        gp  = fn(d["gaze"]);  moved += int(gp != gaze_patch_raw(d["gaze"]))
        top = int(cand_idx[d["drops"][cand_idx].argmax()])
        gr, gc = divmod(gp, G); tr, tc = divmod(top, G)
        dist.append(math.hypot(tr-gr, tc-gc)); offs.append((tc-gc, tr-gr))
    dist = np.array(dist); dx, dy = zip(*offs)
    chance = float(np.mean([[math.hypot(i//G - divmod(fn(d["gaze"]), G)[0],
                                        i%G  - divmod(fn(d["gaze"]), G)[1])
                             for i in cand_idx.tolist()] for d in data]))
    print(f"{label}")
    print(f"   median {np.median(dist):.2f}   mean {dist.mean():.2f}  vs chance {chance:.2f}")
    print(f"   within fovea (<=1.5) {(dist<=1.5).sum()}/{len(dist)}   "
          f"beyond 3 cells {(dist>3).sum()}/{len(dist)}")
    print(f"   offset spread: col sigma {np.std(dx):.2f}, row sigma {np.std(dy):.2f}  "
          f"| mean ({np.mean(dx):+.2f}, {np.mean(dy):+.2f})")
    if moved:
        print(f"   gaze patch changed for {moved}/{len(dist)} examples")
    return dist, np.array(dx), np.array(dy)

d_raw, *_ = summarise(gaze_patch_raw, "AS REPORTED (no letterbox correction)")
print()
d_fix, dx, dy = summarise(gaze_patch_fixed, "CORRECTED")
GP = gaze_patch_fixed if PADDED else gaze_patch_raw

fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.2))
ax[0].hist([d_raw, d_fix], bins=np.arange(0, G+1, 1.0), label=["as reported", "corrected"])
ax[0].legend(fontsize=8); ax[0].set_xlabel("gaze -> top-drop distance (cells)")
ax[0].set_title("Did the correction change anything?")
ax[1].scatter(dx, dy, s=45, alpha=.75); ax[1].scatter([0], [0], marker="x", s=160, c="lime", linewidths=3)
ax[1].axhline(0, lw=.5, c="k"); ax[1].axvline(0, lw=.5, c="k")
ax[1].set_xlim(-G, G); ax[1].set_ylim(G, -G)
ax[1].set_xlabel("column offset"); ax[1].set_ylabel("row offset")
ax[1].set_title("Corrected offsets (clustered = learnable)")
plt.tight_layout(); plt.show()

## 5. CHECK B — headroom

The best geometric predictor available from gaze alone: an anisotropic Gaussian using the measured
offset spread, optionally shifted by the measured mean offset. If this matches the best learned
teacher, a module that predicts relevance from gaze has nothing left to add.

In [ ]:
sig_c, sig_r = max(float(np.std(dx)), .5), max(float(np.std(dy)), .5)
mu_c,  mu_r  = float(np.mean(dx)), float(np.mean(dy))
print(f"fitted blob: sigma_col {sig_c:.2f}, sigma_row {sig_r:.2f}, "
      f"mean offset ({mu_c:+.2f}, {mu_r:+.2f})\n")

def blob(gp, sc, sr, oc=0.0, orr=0.0):
    r0, c0 = divmod(gp, G); r0, c0 = r0 + orr, c0 + oc
    return torch.tensor([-(((i//G - r0)/sr)**2 + ((i%G - c0)/sc)**2) for i in range(L_v)])

def isotropic(gp):
    r0, c0 = divmod(gp, G)
    return torch.tensor([-math.hypot(i//G - r0, i % G - c0) for i in range(L_v)])

g = torch.Generator().manual_seed(0)
PREDICTORS = {
    "anisotropic gaze blob":  lambda d: blob(GP(d["gaze"]), sig_c, sig_r),
    "  + mean-offset shift":  lambda d: blob(GP(d["gaze"]), sig_c, sig_r, mu_c, mu_r),
    "isotropic gaze prox":    lambda d: isotropic(GP(d["gaze"])),
    "center (no gaze)":       lambda d: isotropic(L_v // 2),
    "CTRL random":            lambda d: torch.rand(L_v, generator=g),
}
QUOTED = {"attn x grad (teacher)": .350, "attention imp_a (teacher)": .311}

def topk_cand(v, k):
    return set(cand_idx[torch.topk(v[cand_idx], k).indices].tolist())

from scipy.stats import norm
for k in (5, 10, 12):
    chance, rows, used = k / n_cand, defaultdict(list), 0
    for d in data:
        if float(torch.topk(d["drops"][cand_idx], k).values[-1]) <= 1e-6:
            continue
        used += 1
        gt = topk_cand(d["drops"], k)
        for name, f in PREDICTORS.items():
            rows[name].append(len(topk_cand(f(d), k) & gt) / k)
    print(f"\n=== precision@{k}   chance {chance:.1%}   usable {used}/{N}")
    scored = {n: float(np.mean(v)) for n, v in rows.items()}
    if k == 10:
        scored.update(QUOTED)
    for name in sorted(scored, key=lambda n: -scored[n]):
        extra = ""
        if name in rows:
            a = np.array(rows[name]); se = a.std(ddof=1)/max(np.sqrt(len(a)), 1e-9)
            p = 2*(1-norm.cdf(abs((a.mean()-chance)/se))) if se > 0 else 1.0
            extra = f"{p:>8.3f}{'  *' if p < 0.05 else ''}"
        else:
            extra = "   (quoted)"
        print(f"{name:<28}{scored[name]:>8.1%}{scored[name]/chance:>9.2f}x{extra}")

## 6. Verdict

**Check A**
* *SQUASHED* -> Check 2 stands as reported; the horizontal offset spread is a real property.
* *PADDED* -> every gaze distance in the gaze eval and in Check 2 was wrong. Use the corrected
  numbers above, and note that the "context is displaced sideways" reading may have been the pad.

**Check B** — compare the anisotropic gaze blob against the teachers at k=10:
* **>= ~31%** -> a 3-parameter geometric rule matches the best learned teacher. **FRM has no
  headroom**; the FRM spec's own kill criterion fires ("FRM ~ Eccentricity -> DROP FRM"). Stop before
  generating labels.
* **~20-25%** -> the blob beats isotropic proximity but leaves a clear gap to the teachers. That gap
  is what a learned FRM would have to capture. Proceed to LOO-at-scale.
* **~= isotropic (19%)** -> anisotropy adds nothing; gaze carries little spatial information about
  where the context is, and FRM would be learning from a weak input.

Caveat that applies either way: n=18-20, and none of these are paired tests. Treat a gap under ~5
points as unresolved and raise `N_PER_TYPE` before acting on it.